# 04 · BERT and Legal-BERT (3 seeds each)

Phase 6. Needs a GPU (T4). Each seed takes a while; finished seeds are saved to Drive, so after a disconnect re-run only the missing seed (e.g. `--seeds 44`).

Set `MAX_LEN` from the token-length percentiles in `01_eda.ipynb`.

## Setup
Clones the repo, installs packages, and links `models/` and `mlruns/` to Google Drive so nothing is lost if Colab disconnects.

In [ ]:
# 1. Get the code (for a private repo, see the README: add GITHUB_TOKEN in Colab Secrets)
import os
if not os.path.exists("/content/fineguard"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        !git clone -q https://{token}@github.com/424103-achu/fineguard.git /content/fineguard
    except Exception:
        !git clone -q https://github.com/424103-achu/fineguard.git /content/fineguard
%cd /content/fineguard/ml
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 2. Save outputs to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/fineguard"
!mkdir -p {DRIVE}/models {DRIVE}/mlruns {DRIVE}/figures
!rm -rf models mlruns && ln -sfn {DRIVE}/models models && ln -sfn {DRIVE}/mlruns mlruns

# MLflow's SQLite file works best on local disk: copy it from Drive now, back at the end
import os, shutil
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:////content/mlflow.db"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
if os.path.exists(f"{DRIVE}/mlruns/mlflow.db"):
    shutil.copy(f"{DRIVE}/mlruns/mlflow.db", "/content/mlflow.db")
FIG = f"{DRIVE}/figures"

In [ ]:
MAX_LEN = 128   # from 01_eda: 95th/99th percentile of Legal-BERT tokens

## BERT (`bert-base-uncased`)

In [ ]:
!python -m src.train_transformer --model-name bert-base-uncased --seeds 42 43 44 --max-len {MAX_LEN}

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md

## Legal-BERT (`nlpaueb/legal-bert-base-uncased`)
The seed with the best **validation** macro-F1 is exported to `models/legal-bert-unfair/` for the web app.

In [ ]:
!python -m src.train_transformer --model-name nlpaueb/legal-bert-base-uncased --seeds 42 43 44 --max-len {MAX_LEN} --export-dir models/legal-bert-unfair

In [ ]:
# Copy the MLflow database back to Drive (run this after every training cell you care about)
shutil.copy("/content/mlflow.db", f"{DRIVE}/mlruns/mlflow.db")
!python -m src.results_table --out {DRIVE}/results.md

## Optional: DistilBERT (faster, for the speed row)

In [ ]:
# !python -m src.train_transformer --model-name distilbert-base-uncased --seeds 42 43 44 --max-len {MAX_LEN} --export-dir models/distilbert-unfair

## Error analysis for the exported Legal-BERT model (Phase 7)

In [ ]:
import numpy as np, pandas as pd
from src.data import load_all_splits
from src.predictor import load_predictor
from src.evaluate import evaluate, per_label_report
from src.error_analysis import error_table, confusion_pairs, f1_by_length

data, LABELS = load_all_splits()
te_x, te_y = data["test"]
predictor = load_predictor("models/legal-bert-unfair")
assert predictor.labels == LABELS
probs, pred = predictor.predict(te_x)
print(evaluate(te_y, pred))
print(per_label_report(te_y, pred, LABELS))

In [ ]:
display(f1_by_length(te_x, te_y, pred))
display(confusion_pairs(te_y, pred, LABELS))
pd.set_option("display.max_colwidth", 250)
error_table(te_x, te_y, pred, LABELS, k=5)

## Optional: LIME explanations

In [ ]:
# !pip install -q lime
from lime.lime_text import LimeTextExplainer
explainer = LimeTextExplainer(class_names=LABELS)
idx = int(np.where(te_y.any(axis=1))[0][0])
j = int(te_y[idx].argmax())
exp = explainer.explain_instance(te_x[idx], lambda xs: predictor.predict_probs(list(xs)),
                                 labels=[j], num_features=8, num_samples=500)
print(te_x[idx]); print(LABELS[j], exp.as_list(label=j))

## CPU latency per document (speed column)
Run this cell on a **CPU** runtime for the `CPU ms / doc` column. It uses your collected ToS documents in `data/tos_docs/` if present.

In [ ]:
import time, glob, torch
from src.pipeline import split_sentences
docs = sorted(glob.glob("data/tos_docs/*.txt"))
if docs:
    cpu_predictor = load_predictor("models/legal-bert-unfair")
    cpu_predictor.model.to("cpu"); cpu_predictor.device = "cpu"
    times = []
    for path in docs:
        text = open(path, encoding="utf-8").read()
        t0 = time.perf_counter(); cpu_predictor.predict(split_sentences(text)); times.append(time.perf_counter() - t0)
    print(f"{len(docs)} docs, mean {1000*np.mean(times):.0f} ms/doc on CPU")
else:
    print("No documents in data/tos_docs/ yet")